<a href="https://colab.research.google.com/github/xwang335/Campbell-A/blob/main/RF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# install cuml
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py

In [ ]:
# verify cuml installation
from cuml.ensemble import RandomForestRegressor as cuRF
print("cuML installed")

import cuml
print(f"cuML version: {cuml.__version__}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import time
import gc
import warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from cuml.ensemble import RandomForestRegressor
import cupy as cp
# from sklearn.ensemble import RandomForestRegressor

In [ ]:
cd /content/drive/MyDrive

/content/drive/MyDrive


In [ ]:
p_path     = "/content/drive/MyDrive/preprocess_data.parquet"
LOCAL_PATH = '/content/df_processed.parquet'

# preprocess and saved to disk
if not os.path.exists(LOCAL_PATH):
    df = pd.read_parquet(p_path)

    # preprocess
    id_cols    = ['DATE', 'permno']
    float_cols = [c for c in df.columns if c not in id_cols]
    df[float_cols] = df[float_cols].astype(np.float32)

    # generate dummy variables
    sic_df        = pd.get_dummies(df['sic2'], prefix='sic')
    sic_cols_list = sic_df.columns.tolist()
    df            = pd.concat([df, sic_df], axis=1)


    # saved to disk
    df.to_parquet(LOCAL_PATH, index=False)
    print(f"finished preprocessing, saved to local disk")

# skip preprocess
else:
    df            = pd.read_parquet(LOCAL_PATH)
    sic_cols_list = [c for c in df.columns if c.startswith('sic_')]
    print(f"read from local disk, skip preprocessing")

print(f"df shape: {df.shape}")

finished preprocessing, saved to local disk
df shape: (3712808, 183)


In [ ]:
def generate_920_features(df, char_cols, macro_cols, sic_cols):

    X_char  = df[char_cols].to_numpy(dtype=np.float32, copy=False)
    X_macro = df[macro_cols].to_numpy(dtype=np.float32, copy=False)
    X_sic   = df[sic_cols].to_numpy(dtype=np.float32, copy=False)

    # 8macro features + const：(N, 9)
    ones         = np.ones((len(df), 1), dtype=np.float32)
    X_macro_aug  = np.hstack([ones, X_macro])   # (N, 9)

    # interaction：94 × 9 = 846
    X_inter = (X_char[:, :, np.newaxis] * X_macro_aug[:, np.newaxis, :]).reshape(len(df), -1)

    # 846 + 74 = 920
    X_920 = np.hstack([X_inter, X_sic])

    return X_920

In [ ]:
macro=['tbl','d/p','e/p','b/m','tms','dfy','ntis','svar']
features=list(df.columns)[2:96]

In [ ]:
def calc_oos_r2(actual, predicted):
    actual    = np.array(actual)
    predicted = np.array(predicted)
    denom = np.sum(actual ** 2)
    if denom == 0:
        return np.nan
    return 1 - np.sum((actual - predicted) ** 2) / denom

In [ ]:
def select_best_model(X_train, y_train, X_val, y_val,
                      depth_candidates=[1,2,3,4,5,6],
                      mf_candidates=[3,5,10,20,30,50],
                      n_estimators=300,
                      n_estimators_screen=50):

    # filter all combinations with 50 trees
    screen_results = []

    X_tr = cp.array(X_train) if not hasattr(X_train, 'device') else X_train
    y_tr = cp.array(y_train) if not hasattr(y_train, 'device') else y_train
    X_v  = cp.array(X_val)   if not hasattr(X_val,   'device') else X_val
    y_v  = np.array(y_val)

    for mf in mf_candidates:
        for depth in depth_candidates:
            rf = RandomForestRegressor(
                n_estimators=n_estimators_screen,
                max_depth=depth,
                max_features=mf,
                bootstrap=True,
                min_samples_leaf=1,
                random_state=24,
            )
            rf.fit(X_tr, y_tr)
            y_pred_v = cp.asnumpy(rf.predict(X_v))
            r2 = calc_oos_r2(y_v, y_pred_v)
            screen_results.append((r2, depth, mf))

    # keep top3
    screen_results.sort(reverse=True)
    top_candidates = screen_results[:3]
    print(f"  top3 candidates: {[(d,mf) for _,d,mf in top_candidates]}")

    # train with 300 trees focusing on top 3 candidates
    best_r2    = -np.inf
    best_model = None
    best_params = None

    for r2_screen, depth, mf in top_candidates:
        rf = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=depth,
            max_features=mf,
            bootstrap=True,
            min_samples_leaf=1,
            random_state=24,
        )
        rf.fit(X_tr, y_tr)
        y_pred_v = cp.asnumpy(rf.predict(X_v))
        r2 = calc_oos_r2(y_v, y_pred_v)
        print(f"    depth={depth}, mf={mf}: "
              f"screen_R²={r2_screen*100:.4f}%  "
              f"full_R²={r2*100:.4f}%")
        if r2 > best_r2:
            best_r2     = r2
            best_model  = rf
            best_params = (depth, mf)

    print(f"  best: depth={best_params[0]}, "
          f"max_features={best_params[1]}, "
          f"val_R²={best_r2*100:.4f}%")
    return best_model, best_params


In [ ]:
def calc_rf_char_importance(model, X_test, y_test, n_char, n_macro):
    """RF freture importance：R² drop, individual char"""
    X_test_gpu = cp.array(X_test.astype(np.float32))
    y_base = model.predict(X_test_gpu)
    if hasattr(y_base, 'get'):
        y_base = y_base.get()
    r2_base = calc_oos_r2(y_test, y_base)

    char_importance = np.zeros(n_char)
    for j in range(n_char):
        X_perturb = X_test.copy()
        X_perturb[:, j*n_macro:(j+1)*n_macro] = 0.0
        y_perturb = model.predict(cp.array(X_perturb.astype(np.float32)))
        if hasattr(y_perturb, 'get'):
            y_perturb = y_perturb.get()
        char_importance[j] = r2_base - calc_oos_r2(y_test, y_perturb)

    return char_importance, r2_base


def calc_rf_macro_importance(model, X_test, y_test, n_char, n_macro, r2_base):
    """RF freture importance：R² drop, macro features"""
    macro_importance = np.zeros(n_macro)
    for k in range(n_macro):
        X_perturb = X_test.copy()
        indices = [j * n_macro + k for j in range(n_char)]
        X_perturb[:, indices] = 0.0
        y_perturb = model.predict(cp.array(X_perturb.astype(np.float32)))
        if hasattr(y_perturb, 'get'):
            y_perturb = y_perturb.get()
        macro_importance[k] = r2_base - calc_oos_r2(y_test, y_perturb)

    return macro_importance

In [ ]:

# def select_max_depth(X_train, y_train, X_val, y_val,
#                      depth_candidates=[1,2,3,4,5,6],
#                      n_estimators=300,
#                      prev_best_depth=None):

#     search    = depth_candidates   # search in the whole range
#     # if prev_best_depth is not None:
#     #     # warm start
#     #     idx = depth_candidates.index(prev_best_depth) \
#     #           if prev_best_depth in depth_candidates else 0
#     #     lo  = max(0, idx - 2)
#     #     hi  = min(len(depth_candidates) - 1, idx + 2)
#     #     search = depth_candidates[lo: hi + 1]
#     # else:
#     #     search = depth_candidates

#     X_tr = X_train.astype(np.float32)
#     y_tr = y_train.astype(np.float32)
#     X_v  = X_val.astype(np.float32)

#     best_r2, best_depth = -np.inf, search[0]

#     for depth in search:
#         rf = RandomForestRegressor(
#             n_estimators=n_estimators,
#             max_depth=depth,
#             max_features='sqrt',
#             bootstrap=True,
#             min_samples_leaf=1,
#             random_state=24,
#         )
#         rf.fit(X_tr, y_tr)
#         y_pred = rf.predict(X_v)

#         # cuML get cupy array，turning back to numpy
#         if hasattr(y_pred, 'get'):
#             y_pred = y_pred.get()

#         r2 = calc_oos_r2(y_val, y_pred)
#         print(f"    depth={depth}  val R²={r2*100:.4f}%")
#         if r2 > best_r2:
#             best_r2, best_depth = r2, depth

#     return best_depth

In [ ]:
# def select_max_depth(X_train, y_train, X_val, y_val,
#                      depth_candidates=[1, 2, 3, 4, 5, 6],
#                      n_estimators=300,
#                      prev_best_depth=None):

#
#     search = depth_candidates

#     best_r2, best_depth = -np.inf, search[0]

#     for depth in search:
#         rf = RandomForestRegressor(
#             n_estimators=n_estimators,
#             max_depth=depth,
#             max_features='sqrt',
#             bootstrap=True,
#             n_jobs=-1,
#             random_state=42,
#         )
#         rf.fit(X_train, y_train)
#         r2 = calc_oos_r2(y_val, rf.predict(X_val))
#         # print(f"    depth={depth}  val R²={r2*100:.4f}%")
#         if r2 > best_r2:
#             best_r2, best_depth = r2, depth

#     return best_depth

In [ ]:
# def select_max_depth(X_train, y_train, X_val, y_val,
#                      depth_candidates=[1, 2, 3, 4, 5, 6],
#                      n_estimators=300,
#                      prev_best_depth=None):
#     """
#     optimize tree depth using validation set (OOS)

#     """
#     if prev_best_depth is not None:
#         # warm start
#         idx = depth_candidates.index(prev_best_depth) \
#               if prev_best_depth in depth_candidates else 0
#         lo  = max(0, idx - 1)
#         hi  = min(len(depth_candidates) - 1, idx + 1)
#         search = depth_candidates[lo: hi + 1]
#     else:
#         search = depth_candidates

#     best_r2, best_depth = -np.inf, search[0]

#     for depth in search:
#         rf = RandomForestRegressor(n_estimators=n_estimators,max_depth=depth,
#            max_features='sqrt',bootstrap=True,n_jobs=-1,
#             random_state=42,
#         )
#         rf.fit(X_train, y_train)
#         r2 = calc_oos_r2(y_val, rf.predict(X_val))
#         print(f"    depth={depth}  val R²={r2*100:.4f}%")
#         if r2 > best_r2:
#             best_r2, best_depth = r2, depth

#     return best_depth


In [ ]:
start_test_year    = 1987
end_test_year      = 2016
DEPTH_CANDIDATES   = [1,2,3,4,5,6]
N_ESTIMATORS_SEL   = 100      # parameter selection on validation set
N_ESTIMATORS_FINAL = 300        # final model

all_preds      = []
year_r2 = []
feature_importance_all= []
macro_names           = ['const'] + macro
n_char                = len(features)
n_macro               = len(macro) + 1
prev_best_depth = None
dates_all = df['DATE'].values.copy()
y_all     = df['exret_lead1'].to_numpy(dtype=np.float32, copy=True)

del df
gc.collect()

needed_cols = features + macro + sic_cols_list + ['DATE', 'permno', 'exret_lead1', 'mvel1']
needed_cols = list(dict.fromkeys(needed_cols))



feature_names = []
for char in features:
    feature_names.append(f"{char}×const")
    for m in macro:
        feature_names.append(f"{char}×{m}")
for sic in sic_cols_list:
    feature_names.append(sic)

In [ ]:

for year in range(start_test_year, end_test_year + 1):
    print(f"\n--- cope with {year} year ---")
    t0 = time.time()

    train_mask = (dates_all >= pd.Timestamp(1957, 3, 1)) & \
                 (dates_all <= pd.Timestamp(year - 13, 12, 31))
    val_mask   = (dates_all >= pd.Timestamp(year - 12, 1, 1)) & \
                 (dates_all <= pd.Timestamp(year - 1,  12, 31))
    test_mask  = (dates_all >= pd.Timestamp(year, 1, 1)) & \
                 (dates_all <= pd.Timestamp(year, 12, 31))

    df_year  = pq.read_table(LOCAL_PATH, columns=needed_cols).to_pandas()
    df_train = df_year[train_mask].reset_index(drop=True)
    df_val   = df_year[val_mask].reset_index(drop=True)
    df_test  = df_year[test_mask].reset_index(drop=True)
    del df_year
    gc.collect()

    X_train = generate_920_features(df_train, features, macro, sic_cols_list)
    y_train = y_all[train_mask]
    X_val   = generate_920_features(df_val,   features, macro, sic_cols_list)
    y_val   = y_all[val_mask]
    X_test  = generate_920_features(df_test,  features, macro, sic_cols_list)
    del df_train, df_val
    gc.collect()

    t1 = time.time()

    # parameter selection and train
    final_model, best_params = select_best_model(
        X_train, y_train, X_val, y_val,
        depth_candidates=[1, 2, 3, 4, 5, 6],
        mf_candidates=[3, 5, 10, 20, 30, 50],
        n_estimators=300,
        n_estimators_screen=50
    )
    del X_val, y_val
    gc.collect()

    t2 = time.time()
    print(f"  best_params={best_params}  select+train: {t2-t1:.1f}s")

    # train set R²
    y_train_pred = final_model.predict(cp.array(X_train.astype(np.float32)))
    if hasattr(y_train_pred, 'get'):
        y_train_pred = y_train_pred.get()
    r2_train = calc_oos_r2(y_train, y_train_pred)
    del X_train, y_train
    gc.collect()

    # predict
    y_pred = final_model.predict(cp.array(X_test.astype(np.float32)))
    if hasattr(y_pred, 'get'):
        y_pred = y_pred.get()

    res = df_test[['DATE', 'permno', 'mvel1', 'exret_lead1']].copy()
    res = res.reset_index(drop=True)
    res['y_pred'] = y_pred
    r2_year_all = calc_oos_r2(res['exret_lead1'], res['y_pred'])

    t3 = time.time()
    print(f"  Year {year} train R²: {r2_train*100:+.4f}%  "
          f"test R²: {r2_year_all*100:+.4f}%  "
          f"predict: {t3-t2:.1f}s")

    #  feature importance：R² drop

    y_te_np   = df_test['exret_lead1'].to_numpy()
    char_importance, r2_base = calc_rf_char_importance(
        final_model, X_test, y_te_np, n_char, n_macro
    )
    macro_importance = calc_rf_macro_importance(
        final_model, X_test, y_te_np, n_char, n_macro, r2_base
    )

    # cope with negative value
    char_importance  = np.maximum(char_importance,  0)
    macro_importance = np.maximum(macro_importance, 0)

    # normalization
    char_total  = char_importance.sum()
    macro_total = macro_importance.sum()
    char_imp_norm  = char_importance  / char_total  if char_total  > 0 \
                     else np.zeros(n_char)
    macro_imp_norm = macro_importance / macro_total if macro_total > 0 \
                     else np.zeros(n_macro)

    t4 = time.time()
    print(f"  importance calc: {t4-t3:.1f}s  |  total: {t4-t0:.1f}s")

    feature_importance_all.append({
        'year':             year,
        'char_importance':  char_imp_norm,
        'macro_importance': macro_imp_norm,
    })

    # annual top10
    top10_idx   = np.argsort(char_imp_norm)[::-1][:10]
    macro_order = np.argsort(macro_imp_norm)[::-1]

    print(f"  Top 10 char features (R² drop):")
    print(f"    {'Rank':<5} {'Feature':<20} {'R²drop share':>12}")
    print(f"    {'-'*39}")
    for rank, idx in enumerate(top10_idx, 1):
        print(f"    {rank:<5} {features[idx]:<20} "
              f"{char_imp_norm[idx]*100:>11.4f}%")

    print(f"  Macro importance (R² drop):")
    print(f"    {'Rank':<5} {'Macro':<10} {'R²drop share':>12}")
    print(f"    {'-'*29}")
    for rank, k in enumerate(macro_order, 1):
        print(f"    {rank:<5} {macro_names[k]:<10} "
              f"{macro_imp_norm[k]*100:>11.4f}%")

    del X_test, y_te_np, final_model, df_test
    gc.collect()

    all_preds.append(res)
    year_r2.append({
        'year':       year,
        'r2':         r2_year_all,
        'best_depth': best_params[0],
        'best_mf':    best_params[1]
    })


# report
results = pd.concat(all_preds, ignore_index=True)
r2_all  = calc_oos_r2(results['exret_lead1'], results['y_pred'])

top1000 = (results
           .sort_values(['DATE', 'mvel1'], ascending=[True, False])
           .groupby('DATE', sort=False).head(1000))
r2_top = calc_oos_r2(top1000['exret_lead1'], top1000['y_pred'])

bot1000 = (results
           .sort_values(['DATE', 'mvel1'], ascending=[True, True])
           .groupby('DATE', sort=False).head(1000))
r2_bot = calc_oos_r2(bot1000['exret_lead1'], bot1000['y_pred'])

df_year_r2 = pd.DataFrame(year_r2)

print(f"\n{'='*45}")
print(f"  {'Subsample':<25}  {'OOS R2':>10}")
print(f"{'-'*45}")
print(f"  {'All stocks':<25}  {r2_all*100:>+10.4f}%")
print(f"  {'Top 1000 (largest)':<25}  {r2_top*100:>+10.4f}%")
print(f"  {'Bottom 1000 (smallest)':<25}  {r2_bot*100:>+10.4f}%")
print(f"{'='*45}")
print(df_year_r2.to_string(index=False))
print(f"\nR²>0:    {(df_year_r2['r2'] > 0).mean():.2%}")
print(f"R² mean: {df_year_r2['r2'].mean()*100:+.4f}%")
print(f"R² std:  {df_year_r2['r2'].std()*100:.4f}%")


# feature importance
char_imp_matrix  = np.stack(
    [d['char_importance']  for d in feature_importance_all])  # (30,94)
macro_imp_matrix = np.stack(
    [d['macro_importance'] for d in feature_importance_all])  # (30,9)

char_imp_mean  = char_imp_matrix.mean(axis=0)
char_imp_mean /= char_imp_mean.sum()

macro_imp_mean  = macro_imp_matrix.mean(axis=0)
macro_imp_mean /= macro_imp_mean.sum()


top10_count = np.zeros(n_char, dtype=int)
for d in feature_importance_all:
    top10_idx_yr = np.argsort(d['char_importance'])[::-1][:10]
    top10_count[top10_idx_yr] += 1

char_rank  = np.argsort(char_imp_mean)[::-1]
macro_rank = np.argsort(macro_imp_mean)[::-1]

print(f"\n{'='*62}")
print(f"  RF Top 20 Average Individual Stock Feature Importances Over 30 Years (R²-Drop Method)")
print(f"{'-'*62}")
print(f"  {'Rank':<5} {'Feature':<20} {'Avg R²drop':>12} {'Top10/30':>10}")
print(f"{'-'*62}")
for rank, idx in enumerate(char_rank[:20], 1):
    print(f"  {rank:<5} {features[idx]:<20} "
          f"{char_imp_mean[idx]*100:>11.4f}%  "
          f"{top10_count[idx]:>6}/30")
print(f"{'='*62}")

print(f"\n{'='*45}")
print(f"  RF Importance of 30-Year Average Macroeconomic Variables (R²-Drop Method)")
print(f"{'-'*45}")
print(f"  {'Rank':<5} {'Macro':<15} {'Avg R²drop':>12}")
print(f"{'-'*45}")
for rank, k in enumerate(macro_rank, 1):
    print(f"  {rank:<5} {macro_names[k]:<15} "
          f"{macro_imp_mean[k]*100:>11.4f}%")
print(f"{'='*45}")


--- cope with 1987 year ---
  top3 candidates: [(1, 50), (1, 30), (2, 30)]
    depth=1, mf=50: screen_R²=1.0779%  full_R²=1.0915%
    depth=1, mf=30: screen_R²=0.7696%  full_R²=0.8249%
    depth=2, mf=30: screen_R²=0.6827%  full_R²=0.5334%
  best: depth=1, max_features=50, val_R²=1.0915%
  best_params=(1, 50)  select+train: 127.4s
  Year 1987 train R²: +3.5213%  test R²: +0.0629%  predict: 1.7s
  importance calc: 42.1s  |  total: 177.0s
  Top 10 char features (R² drop):
    Rank  Feature              R²drop share
    ---------------------------------------
    1     nincr                    49.5224%
    2     rd                       18.9589%
    3     divo                     15.3773%
    4     convind                  12.1203%
    5     mom1m                     4.0210%
    6     zerotrade                 0.0000%
    7     baspread                  0.0000%
    8     ms                        0.0000%
    9     stdcf                     0.0000%
    10    stdacc                    0.00

In [ ]:
export_path = '/content/drive/MyDrive/rf_predictions_obey.parquet'
results.to_parquet(export_path, index=False)
print(f"to Google Drive: {export_path}")

In [ ]:
importance_records = []
all_preds_fixed  = []
# fix depth=3, max_features='sqrt'
for year in range(1987, 2017):
    t0 = time.time()

    train_mask = (dates_all >= pd.Timestamp(1957, 3, 1)) & \
                 (dates_all <= pd.Timestamp(year - 13, 12, 31))
    test_mask  = (dates_all >= pd.Timestamp(year, 1, 1)) & \
                 (dates_all <= pd.Timestamp(year, 12, 31))

    df_all_tmp = pd.read_parquet(LOCAL_PATH, columns=needed_cols)
    df_train   = df_all_tmp[train_mask].reset_index(drop=True)
    df_test    = df_all_tmp[test_mask].reset_index(drop=True)
    del df_all_tmp
    gc.collect()

    X_tr = generate_920_features(df_train, features, macro, sic_cols_list)
    y_tr = df_train['exret_lead1'].to_numpy(dtype=np.float32)
    X_te = generate_920_features(df_test,  features, macro, sic_cols_list)
    del df_train
    gc.collect()

    rf = RandomForestRegressor(
        n_estimators=300, max_depth=3,
        max_features='sqrt', bootstrap=True,
        min_samples_leaf=1, random_state=24
    )
    rf.fit(cp.array(X_tr), cp.array(y_tr))
    del X_tr, y_tr
    gc.collect()

    t1 = time.time()

    # predict
    y_te_pred = rf.predict(cp.array(X_te))
    if hasattr(y_te_pred, 'get'):
        y_te_pred = y_te_pred.get()
    y_te_pred = np.array(y_te_pred)
    y_te_np   = df_test['exret_lead1'].to_numpy()
    r2_year   = calc_oos_r2(y_te_np, y_te_pred)

    t2 = time.time()

    # feature importance：R² drop
    char_importance, r2_base = calc_rf_char_importance(
        rf, X_te, y_te_np, n_char, n_macro
    )
    macro_importance = calc_rf_macro_importance(
        rf, X_te, y_te_np, n_char, n_macro, r2_base
    )

    # cope with negative value and normalization
    char_importance  = np.maximum(char_importance,  0)
    macro_importance = np.maximum(macro_importance, 0)

    char_total  = char_importance.sum()
    macro_total = macro_importance.sum()
    char_imp_norm  = char_importance  / char_total  if char_total  > 0 \
                     else np.zeros(n_char)
    macro_imp_norm = macro_importance / macro_total if macro_total > 0 \
                     else np.zeros(n_macro)

    importance_records.append({
        'year':             year,
        'char_importance':  char_imp_norm,
        'macro_importance': macro_imp_norm,
    })

    t3 = time.time()

    # annual output
    top10_idx   = np.argsort(char_imp_norm)[::-1][:10]
    macro_order = np.argsort(macro_imp_norm)[::-1]

    print(f"\n--- Year {year} ---  R²={r2_year*100:+.4f}%  "
          f"train: {t1-t0:.1f}s  predict: {t2-t1:.1f}s  "
          f"importance: {t3-t2:.1f}s")
    print(f"  Top 10 char features (R² drop):")
    print(f"    {'Rank':<5} {'Feature':<20} {'R²drop share':>12}")
    print(f"    {'-'*39}")
    for rank, idx in enumerate(top10_idx, 1):
        print(f"    {rank:<5} {features[idx]:<20} "
              f"{char_imp_norm[idx]*100:>11.4f}%")
    print(f"  Macro importance (R² drop):")
    print(f"    {'Rank':<5} {'Macro':<10} {'R²drop share':>12}")
    print(f"    {'-'*29}")
    for rank, k in enumerate(macro_order, 1):
        print(f"    {rank:<5} {macro_names[k]:<10} "
              f"{macro_imp_norm[k]*100:>11.4f}%")

    res = df_test[['DATE', 'permno', 'mvel1', 'exret_lead1']].copy()
    res['y_pred'] = y_te_pred
    all_preds_fixed.append(res)
    year_r2.append({'year': year, 'r2': r2_year})

    del X_te, rf, df_test
    gc.collect()


# report
results_fixed = pd.concat(all_preds_fixed, ignore_index=True)
r2_all = calc_oos_r2(results_fixed['exret_lead1'], results_fixed['y_pred'])

top1000 = (results_fixed
           .sort_values(['DATE', 'mvel1'], ascending=[True, False])
           .groupby('DATE', sort=False).head(1000))
r2_top = calc_oos_r2(top1000['exret_lead1'], top1000['y_pred'])

bot1000 = (results_fixed
           .sort_values(['DATE', 'mvel1'], ascending=[True, True])
           .groupby('DATE', sort=False).head(1000))
r2_bot = calc_oos_r2(bot1000['exret_lead1'], bot1000['y_pred'])

df_year_r2 = pd.DataFrame(year_r2)

print(f"\n{'='*45}")
print(f"  {'Subsample':<25}  {'OOS R2':>10}")
print(f"{'-'*45}")
print(f"  {'All stocks':<25}  {r2_all*100:>+10.4f}%")
print(f"  {'Top 1000 (largest)':<25}  {r2_top*100:>+10.4f}%")
print(f"  {'Bottom 1000 (smallest)':<25}  {r2_bot*100:>+10.4f}%")
print(f"{'='*45}")
print(f"\nR²>0:     {(df_year_r2['r2'] > 0).mean():.2%}")
print(f"R² mean:  {df_year_r2['r2'].mean()*100:+.4f}%")
print(f"R² std:   {df_year_r2['r2'].std()*100:.4f}%")


# feature importance
char_imp_matrix  = np.stack(
    [r['char_importance']  for r in importance_records])  # (30,94)
macro_imp_matrix = np.stack(
    [r['macro_importance'] for r in importance_records])  # (30,9)

char_imp_mean  = char_imp_matrix.mean(axis=0)
char_imp_mean /= char_imp_mean.sum()

macro_imp_mean  = macro_imp_matrix.mean(axis=0)
macro_imp_mean /= macro_imp_mean.sum()

top10_count = np.zeros(n_char, dtype=int)
for r in importance_records:
    top10_idx_yr = np.argsort(r['char_importance'])[::-1][:10]
    top10_count[top10_idx_yr] += 1

char_rank  = np.argsort(char_imp_mean)[::-1]
macro_rank = np.argsort(macro_imp_mean)[::-1]

print(f"\n{'='*62}")
print(f"  RF Top 20 Individual Stock Feature Importances (30-Year Average) — R² Drop Method (Depth=3)")
print(f"{'-'*62}")
print(f"  {'Rank':<5} {'Feature':<20} {'Avg R²drop':>12} {'Top10/30':>10}")
print(f"{'-'*62}")
for rank, idx in enumerate(char_rank[:20], 1):
    print(f"  {rank:<5} {features[idx]:<20} "
          f"{char_imp_mean[idx]*100:>11.4f}%  "
          f"{top10_count[idx]:>6}/30")
print(f"{'='*62}")

print(f"\n{'='*47}")
print(f"  RF Importance of 30-Year Average Macroeconomic Variables (R² Drop Method, depth=3)")
print(f"{'-'*47}")
print(f"  {'Rank':<5} {'Macro':<15} {'Avg R²drop':>12}")
print(f"{'-'*47}")
for rank, k in enumerate(macro_rank, 1):
    print(f"  {rank:<5} {macro_names[k]:<15} "
          f"{macro_imp_mean[k]*100:>11.4f}%")
print(f"{'='*47}")


--- Year 1987 ---  R²=+0.4563%  train: 13.3s  predict: 0.3s  importance: 43.7s
  Top 10 char features (R² drop):
    Rank  Feature              R²drop share
    ---------------------------------------
    1     sin                      31.5637%
    2     nincr                    13.9694%
    3     ms                        8.1063%
    4     convind                   5.5946%
    5     divo                      4.8076%
    6     baspread                  3.8553%
    7     retvol                    2.6413%
    8     mom36m                    2.2726%
    9     mom1m                     1.6546%
    10    dolvol                    1.6546%
  Macro importance (R² drop):
    Rank  Macro      R²drop share
    -----------------------------
    1     dfy            37.2079%
    2     svar           22.9554%
    3     const          17.2886%
    4     tbl            13.8414%
    5     e/p             8.7067%
    6     b/m             0.0000%
    7     tms             0.0000%
    8     ntis        

In [ ]:
export_path = '/content/drive/MyDrive/rf_predictions_fixed(3).parquet'
results_fixed .to_parquet(export_path, index=False)
print(f"to Google Drive: {export_path}")

to Google Drive: /content/drive/MyDrive/rf_predictions_fixed(3).parquet


In [ ]:
# needed_cols = features + macro + sic_cols_list + ['DATE', 'permno', 'exret_lead1', 'mvel1']
# needed_cols = list(dict.fromkeys(needed_cols))

# for year in range(start_test_year, end_test_year + 1):
#     print(f"\n--- cope with {year} year ---")
#     t0 = time.time()

#     # split data
#     train_mask = (dates_all >= pd.Timestamp(1957, 3, 1)) & \
#                  (dates_all <= pd.Timestamp(year - 13, 12, 31))
#     val_mask   = (dates_all >= pd.Timestamp(year - 12, 1, 1)) & \
#                  (dates_all <= pd.Timestamp(year - 1,  12, 31))
#     test_mask  = (dates_all >= pd.Timestamp(year, 1, 1)) & \
#                  (dates_all <= pd.Timestamp(year, 12, 31))

#     # filter
#     df_year  = pq.read_table(LOCAL_PATH, columns=needed_cols).to_pandas()
#     df_train = df_year[train_mask].reset_index(drop=True)
#     df_val   = df_year[val_mask].reset_index(drop=True)
#     df_test  = df_year[test_mask].reset_index(drop=True)
#     del df_year
#     gc.collect()

#     # generate features
#     X_train = generate_920_features(df_train, features, macro, sic_cols_list)
#     y_train = y_all[train_mask]
#     X_val   = generate_920_features(df_val,   features, macro, sic_cols_list)
#     y_val   = y_all[val_mask]
#     X_test  = generate_920_features(df_test,  features, macro, sic_cols_list)

#     del df_train, df_val
#     gc.collect()

#     t1 = time.time()
#     # print(f'  feature construction: {t1-t0:.1f}s  '
#     #       f'| train: {len(X_train):,}  val: {len(X_val):,}')

#     # parameter selection


#     best_depth = select_max_depth( X_train, y_train, X_val, y_val, depth_candidates=DEPTH_CANDIDATES,

#       n_estimators=N_ESTIMATORS_SEL,prev_best_depth=prev_best_depth, )

#     t2 = time.time()
#     print(f"  best_depth={best_depth}  select parameter: {t2-t1:.1f}s")

#     # combine train and validation set
#     X_trainval = np.vstack([X_train, X_val])
#     # y_trainval = np.concatenate([y_train, y_val])

#     final_model = RandomForestRegressor(
#         n_estimators=N_ESTIMATORS_FINAL,
#         max_depth=best_depth,
#         max_features='sqrt',
#         bootstrap=True,
#         min_samples_leaf=3,
#         random_state=24,
#     )
#     # 训练用 winsorize 后的值
#     lower = np.percentile(y_train, 0.5)
#     upper = np.percentile(y_train, 99.5)
#     y_train_w = np.clip(y_train, lower, upper)
#     y_val_w   = np.clip(y_val,   lower, upper)

#     # 模型训练用 winsorize 后的值
#     final_model.fit(X_trainval, np.concatenate([y_train_w, y_val_w]))

#     # final_model.fit(X_trainval, y_trainval)
#     del X_trainval, y_train_w, y_val_w
#     gc.collect()

#     t3 = time.time()
#     print(f"  train: {t3-t2:.1f}s  |  total: {t3-t0:.1f}s")

#     # predict
#     res = df_test[['DATE', 'permno', 'mvel1','exret_lead1']].copy().reset_index(drop=True)
#     y_train_pred = final_model.predict(X_train.astype(np.float32))
#     if hasattr(y_train_pred, 'get'):
#         y_train_pred = y_train_pred.get()
#     r2_train = calc_oos_r2(y_train, y_train_pred)
#     y_pred = final_model.predict(X_test.astype(np.float32))
#     if hasattr(y_pred, 'get'):
#       y_pred = y_pred.get()
#     res['y_pred'] = y_pred
#     del X_train, X_val, y_train, y_val
#     del X_test, final_model, df_test
#     gc.collect()

#     all_preds.append(res)
#     r2_year_all = calc_oos_r2(res['exret_lead1'], res['y_pred'])
#     # print(f"  Year {year} R2: {r2_year_all*100:>+7.4f}%")
#     print(f"  Year {year} train set R²: {r2_train*100:+.4f}%  Year {year} test set R²: {r2_year_all*100:+.4f}%")
#     year_r2.append({'year': year, 'r2': r2_year_all, 'best_depth': best_depth})
#     prev_best_depth = best_depth


# # report

# results = pd.concat(all_preds, ignore_index=True)

# r2_all = calc_oos_r2(results['exret_lead1'], results['y_pred'])

# top1000 = (
#     results
#     .sort_values(['DATE', 'mvel1'], ascending=[True, False])
#     .groupby('DATE', sort=False).head(1000)
# )
# r2_top = calc_oos_r2(top1000['exret_lead1'], top1000['y_pred'])

# bot1000 = (
#     results
#     .sort_values(['DATE', 'mvel1'], ascending=[True, True])
#     .groupby('DATE', sort=False).head(1000)
# )
# r2_bot = calc_oos_r2(bot1000['exret_lead1'], bot1000['y_pred'])

# print(f"\n{'='*45}")
# print(f"  {'Subsample':<25}  {'OOS R2':>10}")
# print(f"{'-'*45}")
# print(f"  {'All stocks':<25}  {r2_all*100:>+10.4f}%")
# print(f"  {'Top 1000 (largest)':<25}  {r2_top*100:>+10.4f}%")
# print(f"  {'Bottom 1000 (smallest)':<25}  {r2_bot*100:>+10.4f}%")
# print(f"{'='*45}")

# df_year_r2 = pd.DataFrame(year_r2)
# print(df_year_r2.to_string(index=False))
# print(f"\ndf_year_r2>0: {(df_year_r2['r2'] > 0).mean():.2%}")
# print(f"R² mean: {df_year_r2['r2'].mean()*100:+.4f}%")
# print(f"R² s.d.: {df_year_r2['r2'].std()*100:.4f}%")

# print(df_year_r2[['year', 'best_depth']])

In [ ]:
# # 检查 Top 1000 的预测值分布
# top1000_check = (
#     results_d1
#     .sort_values(['DATE', 'mvel1'], ascending=[True, False])
#     .groupby('DATE', sort=False).head(1000)
# )
# print(f"Top 1000 y_pred 均值: {top1000_check['y_pred'].mean():.6f}")
# print(f"Top 1000 y_pred 标准差: {top1000_check['y_pred'].std():.6f}")
# print(f"All stocks y_pred 均值: {results_d1['y_pred'].mean():.6f}")
# print(f"All stocks y_pred 标准差: {results_d1['y_pred'].std():.6f}")

# # value-weighted 回测用的是 mvel1 加权
# # 如果大盘股的预测值集中在很窄的范围，decile 分组会不稳定
# top1000_check['pred_decile'] = pd.qcut(
#     top1000_check['y_pred'], 10, labels=False, duplicates='drop'
# )
# print(f"\nTop 1000 预测值分位数分布:")
# print(top1000_check['pred_decile'].value_counts().sort_index())

Top 1000 y_pred 均值: 0.005908
Top 1000 y_pred 标准差: 0.002003
All stocks y_pred 均值: 0.005844
All stocks y_pred 标准差: 0.002002

Top 1000 预测值分位数分布:
pred_decile
0    36262
1    36085
2    35745
3    35908
4    36004
5    36002
6    35994
7    36000
8    36000
9    36000
Name: count, dtype: int64


In [ ]:
# export
export_path = '/content/drive/MyDrive/rf_predictions.parquet'
results_d1.to_parquet(export_path, index=False)
print(f"to Google Drive: {export_path}")

to Google Drive: /content/drive/MyDrive/rf_predictions1.parquet


In [ ]:
from google.colab import runtime
runtime.unassign()

In [ ]:
# import cuml
# help(cuml.ensemble.RandomForestRegressor)